In [58]:
import pandas as pd
import pathlib as pl
import re

ref_base_path = pl.Path("/home/ebertp/work/code/cubi/project-chrom-y-extended/annotation/norm")

label_base_path = pl.Path("/home/ebertp/work/projects/chrom-y-extended/wf-data/region-labels/final_annot/2026-01_final")

ref_files = {
    "t2tv2": "HG002-Y_T2Tv2_regions_repeat-details.bed",
    "hg38": "GRCh38-Y_regions_repeat-details.bed"
}


def define_label_order(fpath):

    df = pd.read_csv(fpath, sep="\t", header=0)

    match_selected = re.compile("^[0-9]+[nu]{1}_")
    select_label = lambda label: match_selected.match(label) is not None

    df["select_label"] = df["name"].apply(select_label)
    # for hg38: CEN1 - CEN2 are probably SAT in T2T
    df = df.loc[~df["seqclass"].isin(["CEN1", "CEN2"]), :].copy()
    df = df.loc[df["select_label"], :].copy()
    df = df.drop_duplicates(["seqclass"], keep=False, inplace=False).reset_index(drop=True, inplace=False)
    df.drop(["fasta_header", "select_label"], axis=1, inplace=True)
    return df


def select_sample_labels(fpath, ref_labels):

    df = pd.read_csv(fpath, sep="\t", header=0)
    df = df.loc[df["#seq"].str.endswith("_chrY"), :].copy()
    df["name"] = df["name"].replace({"DYZ19_Yq": "DYZ19", "CEN-DYZ3": "CEN_DYZ3"}, inplace=False)
    
    df["select_label"] = df["name"].isin(ref_labels["seqclass"])
    df = df.loc[df["select_label"], :].reset_index(drop=True, inplace=False)
    df.sort_values("start", inplace=True)
    order_numbers = []
    current_number = 0
    for row in df.itertuples():
        try:
            next_name = df.loc[row.Index+1, "name"]
        except KeyError:
            order_numbers.append(current_number)
            continue  # last row = break
        else:
            if row.name == next_name:
                order_numbers.append(current_number)
                continue
            else:
                order_numbers.append(current_number)
                current_number += 1
    assert len(order_numbers) == df.shape[0]
    df["order_num"] = order_numbers
    df["shift_num"] = order_numbers
    return df


def debug(name, df, direct):
    print("shifting ", direct)
    row = df.loc[df["name"] == name, :]
    print(row)
    return


def define_label_order_matching(sample, sample_labels, ref, ref_labels):


    collect = []
    
    for ref_label in ref_labels.itertuples():
        matches = sorted(set(sample_labels.loc[sample_labels["name"] == ref_label.seqclass, "shift_num"].values))
        if len(matches) == 0:
            collect.append((sample, ref, "MISS", ref_label.seqclass, ref_label.Index, -1, 0))
            # shift all order numbers larger this one by one
            # to account for missing/gap in ordering
            higher_order = sample_labels["order_num"] > ref_label.Index
            sample_labels.loc[higher_order, "shift_num"] += 1
            continue
        for order_num in matches:
            if order_num == int(ref_label.Index):
                collect.append(
                    (sample, ref, "HIT", ref_label.seqclass, ref_label.Index, order_num, 0)
                )
            else:
                delta = ref_label.Index - order_num
                # shift by this delta to have all subsequent labels
                # potentially in order
                higher_order = sample_labels["order_num"] > ref_label.Index
                sample_labels.loc[higher_order, "shift_num"] += delta
                collect.append(
                    (sample, ref, "ORDER", ref_label.seqclass, ref_label.Index, order_num, delta)
                )

    collect = pd.DataFrame.from_records(
        collect,
        columns=["sample", "ref", "item", "seqclass", "ref_pos", "sample_pos", "shift"]
    )
    return collect       

collect_stats = []

for ref, ref_file in ref_files.items():
    fp = ref_base_path.joinpath(ref_file)
    ref_labels = define_label_order(fp)

    subfolder = label_base_path.joinpath(ref)

    for region_file in subfolder.glob("*.bed"):
        sample = region_file.name.split(".")[0]
        sample_labels = select_sample_labels(region_file, ref_labels)
        sample_stats = define_label_order_matching(sample, sample_labels, ref, ref_labels)
        collect_stats.append(sample_stats)

collect_stats = pd.concat(collect_stats, axis=0, ignore_index=False)
collect_stats.sort_values(["ref", "sample", "ref_pos"], inplace=True)

collect_stats.to_csv("label-order-stats.tsv", sep="\t", index=False, header=True)


agg = collect_stats.groupby(["ref", "sample", "item"]).size()
agg.to_csv("label-order-agg.tsv", sep="\t", index=True, header=True)
print(agg)
    

ref    sample   item 
hg38   200080   HIT      23
                ORDER     2
       200084   HIT      20
                ORDER     9
       200085   HIT      13
                         ..
t2tv2  NA20870  ORDER     4
       NA20905  HIT       5
                MISS      1
                ORDER    20
       NA21093  HIT      23
Length: 536, dtype: int64
